## 4.4 AMC 自适应 MCS 跟踪与 QoS 流控

在上一节中，我们学习了 HARQ 重传机制。本节先演示 AMC 在 SNR 时变场景下的 MCS 跟踪行为，再演示 QoS 流控背压机制。

本节学习大纲如下：

- LinkQualityTracker 内部实现
- SNR 时变场景下的 MCS 跟踪
- QoS 流控：突发到达与暂停/恢复


---

### 1. AMC 自适应 MCS 跟踪

#### 1.1 该实验涉及的关键文件

```
src/nearlink_sdr/
├── mac/
│   ├── qos.py              <- LinkQualityTracker: 滑动窗口 FER 统计与 MCS 建议
│   └── frame.py            <- AsyncDataFrame: MAC 数据帧组包
├── phy/
│   ├── mac_interface.py    <- mac_to_iq / iq_to_mac: MAC-PHY 转换
│   └── tx_pipeline.py      <- TxConfig: 发射参数配置 (含 MCS 索引)
├── common/
│   └── mcs.py              <- get_mcs: 查询 MCS 对应的调制与码率
└── sim/
    └── link_sim.py          <- _channel_impair: 信道损伤 (AWGN 加噪)
```


---

#### 1.2 LinkQualityTracker 内部实现

In [ ]:
!cat -n src/nearlink_sdr/mac/qos.py | sed -n "305,374p"

### 关键函数说明

上面 `cat` 展示了 `LinkQualityTracker` 的完整源码。以下逐一拆解每个属性和方法的内部逻辑：

---

**数据结构**

- **`_history: deque[bool]`** — 滑动窗口，存储最近 `window_size`（默认 32）帧的 CRC 结果。`deque` 是 Python 内置的双端队列，`.append()` 追加到队尾，`.popleft()` 从队首弹出最旧记录。窗口满时自动踢掉最旧数据。
- **`_current_mcs: int`** — 当前 MCS 等级（0-12），通过 `current_mcs` property 暴露，setter 自动钳位在 [0, 12]。
- **`fer_target_low = 0.01`** — FER 下界：窗口 FER 低于 1% 说明链路质量过剩，可以提升 MCS。
- **`fer_target_high = 0.10`** — FER 上界：窗口 FER 高于 10% 说明链路质量不足，需要降低 MCS。

---

**`record(crc_ok)`** — 将本帧 CRC 结果写入滑动窗口。

```python
def record(self, crc_ok: bool) -> None:
    self._history.append(crc_ok)               # 追加到队尾
    if len(self._history) > self.window_size:   # 超长则踢掉最旧
        self._history.popleft()
```

---

**`fer` (property)** — 实时计算窗口内 FER = 失败帧数 / 窗口帧数。窗口为空时返回 0。

```python
@property
def fer(self) -> float:
    if not self._history:
        return 0.0
    n_fail = sum(1 for ok in self._history if not ok)  # 遍历 deque 数 False
    return n_fail / len(self._history)
```

---

**`suggest_mcs_adjustment()`** — 核心决策函数。根据窗口 FER 和当前 MCS 返回调整方向。注意：数据不足（窗口不满一半）时返回 0 防止无依据的调整。

```python
def suggest_mcs_adjustment(self) -> int:
    if len(self._history) < self.window_size // 2:    # 数据不足 → 不调
        return 0
    if self.fer < self.fer_target_low and self._current_mcs < 12:
        return 1                                       # FER 太低 → 建议升
    if self.fer > self.fer_target_high and self._current_mcs > 0:
        return -1                                      # FER 太高 → 建议降
    return 0                                           # 在目标区间内 → 保持
```


---

**`apply_suggestion()`** — 将 `suggest_mcs_adjustment()` 的结果应用到内部状态。调用一次即完成"决策 + 实施"，`_current_mcs` 被更新。

```python
def apply_suggestion(self) -> int:
    adj = self.suggest_mcs_adjustment()      # 获取 ±1/0
    self.current_mcs = self._current_mcs + adj 
    return self._current_mcs
```

---

**调用关系总结**：

```
record(crc_ok)            → 写入 CRC
fer (property)            → 读数据（遍历 deque 数失败数）
suggest_mcs_adjustment()  → 决策（读 fer，返回 ±1/0）
apply_suggestion()        → 实施（建议 + 钳位 → 更新 _current_mcs）
```

在实际使用中，每帧调用 `record(crc_ok)` + `apply_suggestion()` 即可完成完整的 AMC 闭环——前者记录结果，后者根据窗口 FER 决定是否调整。下面演示这一过程：


In [ ]:
import sys
sys.path.insert(0, "../src")
from nearlink_sdr.mac.qos import LinkQualityTracker

tracker = LinkQualityTracker(window_size=8)
tracker._current_mcs = 5

# 写入 3 个成功 + 5 个失败，观察窗口内 FER 和 MCS 建议
events = [True, True, True, False, False, False, False, False]
for i, crc_ok in enumerate(events):
    tracker.record(crc_ok)
    adj = tracker.suggest_mcs_adjustment()
    tracker.apply_suggestion()
    print(f"frame {i}: CRC={'OK' if crc_ok else 'FAIL'}  "
          f"window={list(tracker._history)}  FER={tracker.fer:.2f}  "
          f"MCS={tracker._current_mcs}  suggest={adj:+d}")


---

#### 1.3 导入与 SNR 时变场景

SNR 按 5 -> 12 -> 3 -> 18 -> 8 dB 跳变，每段持续 20 帧，模拟信道质量波动。

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.common.mcs import get_mcs
from nearlink_sdr.mac.qos import LinkQualityTracker
from nearlink_sdr.phy.mac_interface import iq_to_mac, mac_to_iq
from nearlink_sdr.phy.tx_pipeline import TxConfig
from nearlink_sdr.sim.link_sim import _channel_impair

n_frames_per_snr = 20
snr_sequence = np.concatenate([
    np.full(n_frames_per_snr, 5),
    np.full(n_frames_per_snr, 12),
    np.full(n_frames_per_snr, 3),
    np.full(n_frames_per_snr, 18),
    np.full(n_frames_per_snr, 8),
])

rng = np.random.default_rng(42)
tracker = LinkQualityTracker(window_size=8)
tracker._current_mcs = 5

snr_trace, mcs_trace, fer_trace = [], [], []

for snr in snr_sequence:
    mcs_idx = tracker.current_mcs
    cfg = TxConfig(
        frame_type=2, mcs_index=mcs_idx, pid=0x123456,
        whitening_seed=0x52, crc_seed=0x555555,
        crc_len=24, ctrl_bits_len=28, pilot_interval=8,
    )
    mac_payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    from nearlink_sdr.mac.frame import AsyncDataFrame
    frame = AsyncDataFrame(segment_type=0, data=mac_payload)
    mac_bytes = frame.pack()
    iq = mac_to_iq(mac_bytes, cfg)
    rx_iq = _channel_impair(iq, float(snr), "awgn", 6.0, 0.0, "none", cfg.sps, rng)
    rx = iq_to_mac(rx_iq, cfg, len(mac_bytes))

    tracker.record(rx.crc_ok)
    tracker.apply_suggestion()

    snr_trace.append(float(snr))
    mcs_trace.append(mcs_idx)
    fer_trace.append(tracker.fer)

---

#### 1.4 MCS 跟踪曲线

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
frames = list(range(len(snr_trace)))
ax.plot(frames, snr_trace, "-", alpha=0.5, label="SNR")
ax.set_xlabel("Frame Index"); ax.set_ylabel("SNR (dB)")
ax.set_title("AMC Adaptive MCS Tracking")
ax.legend(loc="upper left"); ax.grid(True, ls="--", alpha=0.5)

ax2 = ax.twinx()
ax2.step(frames, mcs_trace, "r-", where="mid", label="MCS")
ax2.set_ylabel("MCS Index"); ax2.set_yticks(range(13))
ax2.legend(loc="upper right")
plt.show()
print("根据 SNR 曲线变化后 MCS 跟随变化的快慢可对跟踪精度进行分析。")

---

### 2. QOS 流控


#### 2.1 本实验涉及的关键文件

```
src/nearlink_sdr/
├── mac/
│   └── qos.py              <- FlowController / TxQueue / Priority: QoS 流控与发送队列
└── sim/
    └── link_sim.py          <- sim_qos_flow_control: 突发到达流控仿真
```

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "3442,3500p"

关键函数说明：

**`FlowController`** — 流控核心。维护 `buffer_count` 计数器，配合高/低水位线实现背压：

```python
class FlowController:
    buffer_high_watermark: int = 16      # 高水位: 缓冲区上限
    buffer_low_watermark: int = 4        # 低水位: 恢复阈值
    _buffer_count: int = 0               # 当前缓冲计数
    _paused: bool = False                # 暂停标志

    def enqueue(self, count: int = 1) -> None:
        self._buffer_count += count
        if self._buffer_count >= self.buffer_high_watermark:
            self._paused = True          # 达到高水位 → 暂停入队

    def dequeue(self, count: int = 1) -> None:
        self._buffer_count = max(0, self._buffer_count - count)
        if self._paused and self._buffer_count <= self.buffer_low_watermark:
            self._paused = False         # 降至低水位 → 恢复入队

    @property
    def is_paused(self) -> bool:
        return self._paused
```

其中 `buffer_high_watermark` 和 `buffer_low_watermark` 在演示中分别设为 8 和 2。双水位设计的目的是避免在单一门槛附近频繁振荡——类似施密特触发器的迟滞效应。

---

**`TxQueue`** — 发送队列。底层封装 `deque`，`push()` 入队（队列满返回 False），`pop()` 出队，`is_empty` 判断是否为空。

```python
class TxQueue:
    max_size: int = 64
    _items: deque = field(default_factory=deque)

    def push(self, item: TxQueueItem) -> bool:
        if len(self._items) >= self.max_size:
            return False                 # 队列满 → 拒绝入队
        self._items.append(item)
        return True

    def pop(self) -> TxQueueItem | None:
        return self._items.popleft() if self._items else None

    @property
    def is_empty(self) -> bool:
        return len(self._items) == 0
```

---

**`TxQueueItem`** — 队列元素，包含 `priority`（Priority 枚举）和 `data`（bytes 载荷）。发送时按优先级调度，高优先级先发送。

**`Priority`** — 优先级枚举，`NORMAL` 和 `HIGH` 两种，决定发送顺序。

---

**流控闭环**：上层产生数据 → `TxQueue.push(item)` → `FlowController.enqueue()` → 高水位触发暂停 → 下层消费数据 → `TxQueue.pop()` → `FlowController.dequeue()` → 低水位恢复。四个步骤形成完整的背压回路。

#### 2.2 流控演示
每 10 帧产生一批随机数量的数据，当缓冲区 count >= 8 时暂停入队，count <= 2 时恢复：


In [ ]:
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.qos import FlowController, Priority, TxQueue, TxQueueItem
from nearlink_sdr.phy.mac_interface import iq_to_mac, mac_to_iq
from nearlink_sdr.phy.tx_pipeline import TxConfig
from nearlink_sdr.sim.link_sim import _channel_impair
n_frames = 50
burst_size = 10
snr_db = 15.0
high_watermark = 8
low_watermark = 2
rng = np.random.default_rng(42)
cfg = TxConfig(
    frame_type=2, mcs_index=7, pid=0x123456,
    whitening_seed=0x52, crc_seed=0x555555,
    crc_len=24, ctrl_bits_len=28, pilot_interval=8,
)
flow = FlowController(buffer_high_watermark=high_watermark, buffer_low_watermark=low_watermark)
tx_queue = TxQueue(max_size=32)
for i in range(n_frames):
    if i % burst_size == 0:
        for _ in range(rng.integers(3, burst_size + 1)):
            payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
            item = TxQueueItem(priority=Priority.NORMAL, data=payload)
            if tx_queue.push(item):
                flow.enqueue()
    tx_ok = False
    if not tx_queue.is_empty:
        item = tx_queue.pop()
        flow.dequeue()
        frame = AsyncDataFrame(segment_type=0, data=item.data)
        mac_bytes = frame.pack()
        iq = mac_to_iq(mac_bytes, cfg)
        rx_iq = _channel_impair(iq, snr_db, "awgn", 6.0, 0.0, "none", cfg.sps, rng)
        rx = iq_to_mac(rx_iq, cfg, len(mac_bytes))
        tx_ok = rx.crc_ok
    pause_mark = " << PAUSED" if flow.is_paused else ""
    if i % 5 == 0 or flow.is_paused:
        print(f"frame {i:3d}: buf={flow.buffer_count:2d}  TX={'OK' if tx_ok else '--'}  "
              f"paused={'YES' if flow.is_paused else 'no'}{pause_mark}")


----

## 课后实践

请补全下方 AMC 跟踪精度对比中的 **4 处空缺**（每处一行代码），对比不同滑动窗口大小对 MCS 跟踪行为的影响。

要求：

1. 为每个 window_size 创建独立的 LinkQualityTracker
2. 补全 CRC 记录（record）
3. 补全 MCS 决策（apply_suggestion）
4. 补全 MCS 历史记录供绘图

完成后运行 `python amc_window_practice.py`，观察不同窗口大小下的 MCS 跟踪速率差异。

In [ ]:
%%writefile amc_window_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.mac.qos import LinkQualityTracker
from nearlink_sdr.phy.mac_interface import iq_to_mac, mac_to_iq
from nearlink_sdr.phy.tx_pipeline import TxConfig
from nearlink_sdr.sim.link_sim import _channel_impair

# SNR 跳变序列
n_per = 20
snr_sequence = np.concatenate([
    np.full(n_per, 5), np.full(n_per, 12),
    np.full(n_per, 3), np.full(n_per, 18), np.full(n_per, 8),
])
rng = np.random.default_rng(42)
window_sizes = [4, 8, 16, 32]

plt.figure(figsize=(10, 5))
for ws in window_sizes:
    # ==== TODO: 补全 AMC 跟踪（4处空缺）====
    # TODO 1: 为当前 window_size 创建 Tracker
    tracker = ______________
    tracker._current_mcs = 5
    mcs_hist = []

    for snr in snr_sequence:
        mcs_idx = tracker.current_mcs
        cfg = TxConfig(frame_type=2, mcs_index=mcs_idx, pid=0x123456,
                       whitening_seed=0x52, crc_seed=0x555555,
                       crc_len=24, ctrl_bits_len=28, pilot_interval=8)
        mac_payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
        from nearlink_sdr.mac.frame import AsyncDataFrame
        frame = AsyncDataFrame(segment_type=0, data=mac_payload)
        mac_bytes = frame.pack()
        iq = mac_to_iq(mac_bytes, cfg)
        rx_iq = _channel_impair(iq, float(snr), "awgn", 6.0, 0.0, "none", cfg.sps, rng)
        rx = iq_to_mac(rx_iq, cfg, len(mac_bytes))

        # TODO 2: 记录 CRC 结果
        ______________
        # TODO 3: 应用 MCS 调整
        ______________
        # TODO 4: 记录当前 MCS
        ______________

    plt.step(range(len(mcs_hist)), mcs_hist, where="mid",
             label=f"ws={ws}", alpha=0.8)

plt.xlabel("Frame Index"); plt.ylabel("MCS Index")
plt.yticks(range(13))
plt.title("AMC Tracking: window_size Comparison")
plt.legend(); plt.grid(True, ls="--", alpha=0.5); plt.show()

print("窗口越小 → 对 SNR 变化响应越快, 但越容易过冲。")
print("窗口越大 → 响应越平滑, 但滞后越明显。")

执行以下命令进行编译并验证结果：


In [ ]:
!python amc_window_practice.py

预期观察到的结果：窗口越小变化越频繁，窗口越大越稳定但滞后明显。

执行以下代码获取答案


In [ ]:
!cat answer/04.04_answer.txt
